In [5]:
import importnb
import os
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

with importnb.Notebook():
    import acquisition_and_profiling as phase_1
    import scaling_and_3DTransformation as phase_3
    import topology as phase_4

In [6]:


# Pipeline Dependencies
testing_data_path = phase_1.testing_data_path
rul_data_path = phase_1.rul_data_path
subset = phase_1.subset
columns = phase_1.columns
cols_to_drop = phase_1.cols_to_drop

sensor_columns = phase_3.sensor_columns
scaler = phase_3.scaler
WINDOW_SIZE = phase_3.WINDOW_SIZE

device = phase_4.device
model = phase_4.model

# 1. Acquisition & Purging
df_test = pd.read_csv(
    filepath_or_buffer=os.path.join(testing_data_path, f"test_{subset}.txt"),
    sep=r'\s+',
    names=columns,
    index_col=False
)
df_test.set_index(['Engine_ID', 'Cycle'], inplace=True)
df_test.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 2. Strict Scaling (No Data Leakage)
df_test[sensor_columns] = scaler.transform(df_test[sensor_columns])

# 3. 3D Tensor Extraction (Final Window Only)
X_test_list = []
engine_ids_test = df_test.index.get_level_values('Engine_ID').unique()

for engine_id in engine_ids_test:
    engine_data = df_test.xs(engine_id, level='Engine_ID')[
        sensor_columns].values

    # Extract only the final W=30 cycles of the suspended engine
    final_window = engine_data[-WINDOW_SIZE:, :]
    X_test_list.append(final_window)

X_test = np.array(X_test_list)
print(
    f"X_test Shape: {X_test.shape} -> (100 Engines, 30 Cycles, {len(sensor_columns)} Sensors)")

# 4. Target Engineering (The Answer Key)
# Read the true RUL integer values for the 100 test engines
true_rul = pd.read_csv(
    filepath_or_buffer=os.path.join(rul_data_path, f"RUL_{subset}.txt"),
    sep=r'\s+',
    header=None
)[0].values

Y_test = true_rul
print(f"Y_test Shape: {Y_test.shape} -> (100 True Continuous RUL Targets)")

# 5. Inference Execution
# Set the model to evaluation mode (locks gradients)
model.eval()

# Convert numpy array to PyTorch Tensor
tensor_X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

print("\n--- RUNNING INFERENCE ---")
with torch.no_grad():
    # The model outputs raw continuous floating-point numbers
    raw_predictions = model(tensor_X_test)

    # Flatten the 2D output matrix (100, 1) back into a 1D array (100,) for Scikit-Learn
    predictions = raw_predictions.cpu().numpy().flatten()

# 6. The Evaluation Matrix (RMSE & MAE)
print("\n--- REGRESSION PERFORMANCE METRICS ---")

# Mean Absolute Error: The straightforward average of how many cycles we missed by
mae = mean_absolute_error(Y_test, predictions)

# Root Mean Square Error: Heavily penalizes massive deviations (e.g., predicting 150 when true is 20)
rmse = np.sqrt(mean_squared_error(Y_test, predictions))

print(f"Mean Absolute Error (MAE): {mae:.2f} cycles")
print(f"Root Mean Square Error (RMSE): {rmse:.2f} cycles")

# Optional: Print a side-by-side comparison of the first 5 engines for physical verification
print("\n--- SAMPLE PREDICTION VERIFICATION (First 5 Engines) ---")
for i in range(5):
    print(
        f"Engine {i+1} | True RUL: {Y_test[i]} | Predicted RUL: {predictions[i]:.1f} | Error: {abs(Y_test[i] - predictions[i]):.1f}")

X_test Shape: (100, 30, 15) -> (100 Engines, 30 Cycles, 15 Sensors)
Y_test Shape: (100,) -> (100 True Continuous RUL Targets)

--- RUNNING INFERENCE ---

--- REGRESSION PERFORMANCE METRICS ---
Mean Absolute Error (MAE): 75.65 cycles
Root Mean Square Error (RMSE): 86.31 cycles

--- SAMPLE PREDICTION VERIFICATION (First 5 Engines) ---
Engine 1 | True RUL: 112 | Predicted RUL: -0.1 | Error: 112.1
Engine 2 | True RUL: 98 | Predicted RUL: -0.1 | Error: 98.1
Engine 3 | True RUL: 69 | Predicted RUL: -0.2 | Error: 69.2
Engine 4 | True RUL: 82 | Predicted RUL: -0.1 | Error: 82.1
Engine 5 | True RUL: 91 | Predicted RUL: -0.1 | Error: 91.1
